# Part 1: UrbanCart paid-channel analysis

All analysis runs in DuckDB (SQL) directly on the raw CSVs in `../data/`.

In [ ]:
import duckdb

duckdb.sql("SELECT COUNT(*) FROM '../data/users.csv'")

## Step 1: Data exploration (before cleaning)

Profile the raw data and log every problem **before** changing anything, so each issue's "rows affected" count comes from the data as delivered.

In [13]:
# Raw views over the CSVs. Nothing is cleaned here, so every count below reflects the data as delivered.
# DuckDB infers each column's type from its contents; SUMMARIZE shows what it chose.
duckdb.sql("CREATE OR REPLACE VIEW raw_users  AS SELECT * FROM read_csv('../data/users.csv')")
duckdb.sql("CREATE OR REPLACE VIEW raw_orders AS SELECT * FROM read_csv('../data/orders.csv')")
duckdb.sql("CREATE OR REPLACE VIEW raw_spend  AS SELECT * FROM read_csv('../data/marketing_spend.csv')")

CountBIGINT


### 1.1 marketing_spend.csv

In [14]:
duckdb.sql("""
SUMMARIZE raw_spend
""")

column_nameVARCHAR,column_typeVARCHAR,minVARCHAR,maxVARCHAR,approx_uniqueBIGINT,avgVARCHAR,stdVARCHAR,q25VARCHAR,q50VARCHAR,q75VARCHAR,countBIGINT,"null_percentageDECIMAL(9,2)"
month,VARCHAR,2026-01,2026-06,6,NULL,NULL,NULL,NULL,NULL,24,0.00
channel,VARCHAR,Affiliate,Meta Ads,4,NULL,NULL,NULL,NULL,NULL,24,0.00
spend_inr,BIGINT,42000,116000,20,66250.0,20912.342933218213,53000,59000,74000,24,0.00


In [20]:
# Small file: read every row
duckdb.sql("""
SELECT * FROM raw_spend ORDER BY month, channel 
""")

monthVARCHAR,channelVARCHAR,spend_inrBIGINT
2026-01,Affiliate,61000
2026-01,Google Ads,74000
2026-01,Influencer,51000
2026-01,Meta Ads,54000
2026-02,Affiliate,56000
2026-02,Google Ads,59000
2026-02,Influencer,50000
2026-02,Meta Ads,68000
2026-03,Affiliate,59000
2026-03,Google Ads,62000


In [16]:
# Expect each paid channel once per month
duckdb.sql("""
SELECT channel, COUNT(*) AS n_rows
FROM raw_spend
GROUP BY channel
ORDER BY channel
""")

channelVARCHAR,n_rowsBIGINT
Affiliate,6
Google Ads,6
Influencer,6
Meta Ads,6


In [17]:
duckdb.sql("""
SELECT month, COUNT(*) AS n_rows, SUM(spend_inr) AS total_spend
FROM raw_spend
GROUP BY month
ORDER BY month
""")

monthVARCHAR,n_rowsBIGINT,total_spendHUGEINT
2026-01,4,240000
2026-02,4,233000
2026-03,4,249000
2026-04,4,300000
2026-05,4,300000
2026-06,4,268000


### 1.2 users.csv

In [21]:
duckdb.sql("""
SUMMARIZE raw_users
""")                                                                                       

column_nameVARCHAR,column_typeVARCHAR,minVARCHAR,maxVARCHAR,approx_uniqueBIGINT,avgVARCHAR,stdVARCHAR,q25VARCHAR,q50VARCHAR,q75VARCHAR,countBIGINT,"null_percentageDECIMAL(9,2)"
user_id,BIGINT,100001,105135,5188,102568.0,1482.4911466851993,101286,102567,103851,5135,0.00
signup_date,DATE,2026-01-01,2026-06-30,184,2026-04-01 19:32:28.276534,NULL,2026-02-15,2026-04-04,2026-05-17,5135,0.00
city,VARCHAR,Bengaluru,Pune,8,NULL,NULL,NULL,NULL,NULL,5135,0.00
acquisition_channel,VARCHAR,Affiliate,Organic,5,NULL,NULL,NULL,NULL,NULL,5135,1.40
email,VARCHAR,qa.test10@urbancart.in,user105100@gmail.com,5305,NULL,NULL,NULL,NULL,NULL,5135,0.00


In [22]:
duckdb.sql("""
SELECT * FROM raw_users LIMIT 10
""")

user_idBIGINT,signup_dateDATE,cityVARCHAR,acquisition_channelVARCHAR,emailVARCHAR
102343,2026-01-30,Chennai,Meta Ads,user102343@yahoo.in
101432,2026-06-27,Bengaluru,Meta Ads,user101432@gmail.com
104371,2026-04-24,Delhi,Organic,user104371@yahoo.in
102331,2026-04-24,Hyderabad,Meta Ads,user102331@gmail.com
101923,2026-03-09,Pune,Meta Ads,user101923@outlook.com
102461,2026-04-07,Chennai,Affiliate,user102461@outlook.com
104375,2026-02-21,Chennai,Organic,user104375@yahoo.in
102052,2026-06-13,Jaipur,Meta Ads,user102052@outlook.com
102225,2026-04-06,Delhi,Meta Ads,user102225@gmail.com
102324,2026-02-19,Chennai,Meta Ads,user102324@outlook.com


In [23]:
# Exact uniqueness check (SUMMARIZE's approx_unique is only an estimate)
duckdb.sql("""
SELECT COUNT(*)                AS total_rows,
       COUNT(DISTINCT user_id) AS distinct_user_ids,
       COUNT(DISTINCT email)   AS distinct_emails
FROM raw_users
""")

total_rowsBIGINT,distinct_user_idsBIGINT,distinct_emailsBIGINT
5135,5135,5135


In [24]:
# All rows for any repeated user_id: are repeats exact copies or conflicting records?
duckdb.sql("""
SELECT *
FROM raw_users
WHERE user_id IN (SELECT user_id FROM raw_users GROUP BY user_id HAVING COUNT(*) > 1)
ORDER BY user_id
""")

user_idBIGINT,signup_dateDATE,cityVARCHAR,acquisition_channelVARCHAR,emailVARCHAR


In [25]:
# Same check on email: one person under several user_ids?
duckdb.sql("""
SELECT *
FROM raw_users
WHERE email IN (SELECT email FROM raw_users GROUP BY email HAVING COUNT(*) > 1)
ORDER BY email, user_id
""")

user_idBIGINT,signup_dateDATE,cityVARCHAR,acquisition_channelVARCHAR,emailVARCHAR


In [26]:
duckdb.sql("""
SELECT city, COUNT(*) AS n_users
FROM raw_users
GROUP BY city
ORDER BY n_users DESC
""")

cityVARCHAR,n_usersBIGINT
Delhi,690
Hyderabad,653
Jaipur,642
Chennai,641
Mumbai,633
Bengaluru,631
Kolkata,631
Pune,614


In [27]:
# NULLs show up as their own group
duckdb.sql("""
SELECT acquisition_channel, COUNT(*) AS n_users
FROM raw_users
GROUP BY acquisition_channel
ORDER BY n_users DESC
""")

acquisition_channelVARCHAR,n_usersBIGINT
Meta Ads,1300
Affiliate,1235
Google Ads,1100
Organic,828
Influencer,600
NULL,72


In [28]:
# Missing values per column (NULL or blank text)
duckdb.sql("""
SELECT COUNT_IF(user_id IS NULL)                                          AS null_user_id,
       COUNT_IF(signup_date IS NULL)                                      AS null_signup_date,
       COUNT_IF(city IS NULL OR TRIM(city) = '')                          AS missing_city,
       COUNT_IF(acquisition_channel IS NULL OR TRIM(acquisition_channel) = '') AS missing_channel,
       COUNT_IF(email IS NULL OR TRIM(email) = '')                        AS missing_email
FROM raw_users
""")

null_user_idHUGEINT,null_signup_dateHUGEINT,missing_cityHUGEINT,missing_channelHUGEINT,missing_emailHUGEINT
0,0,0,72,0


In [29]:
duckdb.sql("""
SELECT DATE_TRUNC('month', signup_date) AS signup_month, COUNT(*) AS n_users
FROM raw_users
GROUP BY signup_month
ORDER BY signup_month
""")

signup_monthTIMESTAMP,n_usersBIGINT
2026-01-01 00:00:00,861
2026-02-01 00:00:00,800
2026-03-01 00:00:00,808
2026-04-01 00:00:00,911
2026-05-01 00:00:00,908
2026-06-01 00:00:00,847


### 1.3 orders.csv

In [30]:
duckdb.sql("""
SUMMARIZE raw_orders
""")

column_nameVARCHAR,column_typeVARCHAR,minVARCHAR,maxVARCHAR,approx_uniqueBIGINT,avgVARCHAR,stdVARCHAR,q25VARCHAR,q50VARCHAR,q75VARCHAR,countBIGINT,"null_percentageDECIMAL(9,2)"
order_id,BIGINT,500001,505478,5751,502750.7473610134,1575.4751684960622,501396,502763,504106,5684,0.00
user_id,BIGINT,100002,105135,3004,102703.53729767769,1604.125891446872,101210,102764,104124,5684,0.00
order_date,DATE,2026-01-01,2026-06-30,184,2026-04-11 15:18:37.241379,NULL,2026-03-03,2026-04-14,2026-05-22,5684,0.00
order_value,BIGINT,199,3991,1677,1171.2158691062632,534.3095175918462,857,1094,1353,5684,0.00
status,VARCHAR,cancelled,returned,3,NULL,NULL,NULL,NULL,NULL,5684,0.00


In [31]:
duckdb.sql("""
SELECT * FROM raw_orders LIMIT 10
""")

order_idBIGINT,user_idBIGINT,order_dateDATE,order_valueBIGINT,statusVARCHAR
501992,101849,2026-04-03,1056,delivered
500112,100115,2026-02-26,827,cancelled
505404,105127,2026-05-16,2134,delivered
504193,104176,2026-03-22,1487,delivered
504461,104436,2026-04-14,1230,delivered
500928,100781,2026-04-10,1420,delivered
502678,102657,2026-03-24,1349,delivered
502429,102325,2026-03-21,1228,delivered
504139,104146,2026-04-29,999,delivered
502615,102564,2026-04-25,296,returned


In [32]:
duckdb.sql("""
SELECT COUNT(*)                 AS total_rows,
       COUNT(DISTINCT order_id) AS distinct_order_ids
FROM raw_orders
""")

total_rowsBIGINT,distinct_order_idsBIGINT
5684,5478


In [33]:
# All rows for any repeated order_id
duckdb.sql("""
SELECT *
FROM raw_orders
WHERE order_id IN (SELECT order_id FROM raw_orders GROUP BY order_id HAVING COUNT(*) > 1)
ORDER BY order_id
""")

order_idBIGINT,user_idBIGINT,order_dateDATE,order_valueBIGINT,statusVARCHAR
500032,100021,2026-06-29,1156,delivered
500032,100021,2026-06-29,1156,delivered
500116,100116,2026-06-29,1481,delivered
500116,100116,2026-06-29,1481,delivered
500174,100162,2026-03-29,1178,delivered
500174,100162,2026-03-29,1178,delivered
500185,100168,2026-02-26,1327,returned
500185,100168,2026-02-26,1327,returned
500378,100336,2026-05-07,1471,delivered
500378,100336,2026-05-07,1471,delivered


In [34]:
duckdb.sql("""
SELECT status, COUNT(*) AS n_orders
FROM raw_orders
GROUP BY status
ORDER BY n_orders DESC
""")

statusVARCHAR,n_ordersBIGINT
delivered,4949
returned,396
cancelled,339


In [35]:
duckdb.sql("""
SELECT MIN(order_value)                AS min_value,
       MAX(order_value)                AS max_value,
       COUNT_IF(order_value IS NULL)   AS null_value,
       COUNT_IF(order_value <= 0)      AS zero_or_negative
FROM raw_orders
""")

min_valueBIGINT,max_valueBIGINT,null_valueHUGEINT,zero_or_negativeHUGEINT
199,3991,0,0


In [36]:
# Largest orders: plausible, or outliers?
duckdb.sql("""
SELECT * FROM raw_orders ORDER BY order_value DESC LIMIT 10
""")

order_idBIGINT,user_idBIGINT,order_dateDATE,order_valueBIGINT,statusVARCHAR
505305,105115,2026-01-14,3991,delivered
505422,105129,2026-03-04,3970,delivered
505285,105112,2026-04-07,3970,delivered
505424,105129,2026-03-14,3966,delivered
505198,105101,2026-03-17,3965,delivered
505298,105114,2026-03-05,3955,delivered
505341,105119,2026-05-23,3948,delivered
505274,105111,2026-04-04,3945,delivered
505274,105111,2026-04-04,3945,delivered
505248,105108,2026-04-14,3921,delivered


In [37]:
duckdb.sql("""
SELECT DATE_TRUNC('month', order_date) AS order_month, COUNT(*) AS n_orders
FROM raw_orders
GROUP BY order_month
ORDER BY order_month
""")

order_monthTIMESTAMP,n_ordersBIGINT
2026-01-01 00:00:00,513
2026-02-01 00:00:00,824
2026-03-01 00:00:00,1006
2026-04-01 00:00:00,1083
2026-05-01 00:00:00,1176
2026-06-01 00:00:00,1082


### 1.4 Cross-table checks

In [38]:
# Orders whose user_id doesn't exist in users (anti-join)
duckdb.sql("""
SELECT COUNT(*) AS orphan_orders
FROM raw_orders o
LEFT JOIN (SELECT DISTINCT user_id FROM raw_users) u ON o.user_id = u.user_id
WHERE u.user_id IS NULL
""")

orphan_ordersBIGINT
0


In [39]:
# Do channel names match between spend and users? A NULL on one side = name only exists on the other
duckdb.sql("""
SELECT s.channel             AS in_spend,
       u.acquisition_channel AS in_users
FROM (SELECT DISTINCT channel FROM raw_spend) s
FULL OUTER JOIN (SELECT DISTINCT acquisition_channel FROM raw_users) u
  ON s.channel = u.acquisition_channel
ORDER BY in_spend, in_users
""")

in_spendVARCHAR,in_usersVARCHAR
Affiliate,Affiliate
Google Ads,Google Ads
Influencer,Influencer
Meta Ads,Meta Ads
NULL,Organic
NULL,NULL


In [ ]:
# TODO (Aryan): is there another way an order can contradict its user's row? Write that check here.

### 1.5 Findings log

| # | Table | Column | What looks wrong | Rows affected | How I'll handle it |
|---|---|---|---|---|---|
| 1 | | | | | |